# 📖 Notebook 1: Why ID Generation Is Hard

Every record in a database needs a **unique identifier** — a user ID, an order ID, a tweet ID.

When you have one database, this is trivial: `id SERIAL PRIMARY KEY` (auto-increment) just works.

When you have **many servers** generating IDs at the same time, it gets tricky. This notebook shows why.

## Learning Objectives

- Understand why auto-increment IDs become a bottleneck at scale
- See why random UUIDv4 solves coordination but hurts database performance
- Learn what "B-tree index locality" means and why time-ordered IDs matter
- Build intuition for a three-level progression: **BAD → BETTER → BEST**


## 🛠️ Setup

This lab uses **only the Python standard library + `pydantic`**. No Docker, no Redis, no Postgres.

```bash
cd 01-foundations/id-generation
uv sync
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".


## 📏 The setup

Imagine you run a photo-sharing app. Users upload photos and you need to give each photo a unique ID.
You start with one server and one database. Then your app gets popular and you need **many** servers.

Each approach below has a name: we'll call them BAD, BETTER, and BEST — not because one is always right, but because each one **fixes a problem with the previous one**.


## ❌ BAD — Single database auto-increment

The classic approach: `id INTEGER AUTO_INCREMENT`. The database hands out `1, 2, 3, 4, ...`.

**The problem:** every server in your cluster has to call the **same** database to ask for the next number.
That single database becomes a **bottleneck** — it's a single point of failure and a single point of contention.

Let's simulate this with a shared counter protected by a lock (the lock stands in for the database round-trip).


In [ ]:
import threading
import time

class CentralIdService:
    """Stands in for a single-DB auto-increment. Every caller must grab the lock."""
    def __init__(self):
        self._counter = 0
        self._lock = threading.Lock()

    def next_id(self) -> int:
        with self._lock:
            # Simulate network + disk round-trip to the central DB.
            time.sleep(0.0005)  # 0.5 ms
            self._counter += 1
            return self._counter

service = CentralIdService()

def worker(n, out):
    for _ in range(n):
        out.append(service.next_id())

N_WORKERS = 8
PER_WORKER = 200
results = [[] for _ in range(N_WORKERS)]
threads = [threading.Thread(target=worker, args=(PER_WORKER, results[i])) for i in range(N_WORKERS)]

start = time.perf_counter()
for t in threads: t.start()
for t in threads: t.join()
elapsed = time.perf_counter() - start

total_ids = sum(len(r) for r in results)
print(f"Generated {total_ids} IDs in {elapsed:.2f}s")
print(f"Throughput: {total_ids/elapsed:,.0f} IDs/sec")
print(f"First few from worker 0: {results[0][:5]}")


☝️ Notice how throughput is capped no matter how many workers you add. Every worker is queueing on the same lock.

In a real system, the lock is a network + disk round-trip to your primary database. Adding 100 app servers doesn't make it faster — it makes it slower (more contention).


## ⚠️ BETTER — Random UUIDv4 (no coordination)

A UUID (Universally Unique Identifier) is a 128-bit number. Version 4 (`uuid4`) fills it with **random bits**.

Because 128 bits is an enormous space (2¹²⁸ ≈ 3.4 × 10³⁸ possible values), you can safely generate UUIDv4s on *any* machine with **no coordination** and never collide.

This completely solves the bottleneck of BAD.


In [ ]:
import uuid
import time

start = time.perf_counter()
ids = [uuid.uuid4() for _ in range(N_WORKERS * PER_WORKER)]
elapsed = time.perf_counter() - start

print(f"Generated {len(ids)} UUIDv4s in {elapsed*1000:.1f} ms")
print(f"Throughput: {len(ids)/elapsed:,.0f} IDs/sec")
print("First 3:")
for u in ids[:3]:
    print(" ", u)


Much faster — and this scales linearly with machines. No lock, no network round-trip.

### So what's wrong with UUIDv4?

Look at the first few values. They're **random** — no order at all. That breaks something important: **database index locality**.


## 🌳 A mini lesson: why random IDs hurt databases

Most databases (Postgres, MySQL/InnoDB, SQLite) store rows in a **B-tree** sorted by the primary key.

When you insert a new row, the DB finds the right leaf page in the tree and writes the row there.

- **Sequential IDs** (1, 2, 3, ...) → every new row goes into the *same* last page. That page stays hot in memory/cache. Inserts are cheap.
- **Random IDs** (UUIDv4) → every new row lands in a *different* leaf page. The database has to load that page from disk, modify it, write it back. This is called **index fragmentation** and it destroys write performance on big tables.

Let's simulate this with a toy B-tree-ish structure: a dict of "pages", each page holds up to `PAGE_SIZE` keys, sorted.


In [ ]:
import uuid
import random
from bisect import insort

PAGE_SIZE = 16  # keys per page (tiny, for demo)

class ToyBTreeIndex:
    """A ridiculously simplified B-tree leaf layer.

    Pages hold disjoint, sorted key ranges — just like real leaf pages. We count
    'page loads' assuming only ONE page stays hot in memory, so every time an
    insert lands on a different page than the previous insert, that's a disk read.
    """
    def __init__(self):
        self.pages = {0: []}   # page_id -> sorted list of keys
        self.next_page_id = 1
        self.page_loads = 0
        self.last_page_touched = None

    def _touch(self, page_id):
        if page_id != self.last_page_touched:
            self.page_loads += 1
            self.last_page_touched = page_id

    def _page_for(self, key):
        """The leaf page whose range covers `key`.

        That is the page with the smallest max-key that is still >= key. If the key
        is larger than everything in the index, it belongs on the right-most page.
        """
        best = None
        for pid, keys in self.pages.items():
            if not keys:
                return pid
            if keys[-1] >= key and (best is None or keys[-1] < self.pages[best][-1]):
                best = pid
        if best is None:
            best = max(self.pages, key=lambda p: self.pages[p][-1])
        return best

    def insert(self, key):
        pid = self._page_for(key)
        self._touch(pid)
        insort(self.pages[pid], key)
        if len(self.pages[pid]) > PAGE_SIZE:          # split, keeping ranges disjoint
            half = self.pages[pid][PAGE_SIZE // 2:]
            self.pages[pid] = self.pages[pid][:PAGE_SIZE // 2]
            new_pid = self.next_page_id
            self.next_page_id += 1
            self.pages[new_pid] = half

N = 2000

# --- Sequential keys ---
seq_idx = ToyBTreeIndex()
for k in range(N):
    seq_idx.insert(k)

# --- Random keys (UUIDv4-like) ---
rand_idx = ToyBTreeIndex()
rng = random.Random(42)
for _ in range(N):
    rand_idx.insert(rng.random())

# Sanity: this is only a fair comparison if both indexes are actually valid
# B-tree leaf layers, i.e. their page ranges never overlap.
for idx in (seq_idx, rand_idx):
    ranges = sorted((p[0], p[-1]) for p in idx.pages.values() if p)
    assert all(ranges[i][1] < ranges[i + 1][0] for i in range(len(ranges) - 1))

print(f"Sequential inserts: {seq_idx.page_loads:>6} page loads for {N} rows")
print(f"Random inserts:     {rand_idx.page_loads:>6} page loads for {N} rows")
print(f"Random is {rand_idx.page_loads/seq_idx.page_loads:.1f}× worse")

# Sequential inserts keep hitting the same right-most page until it splits, so the
# page-load count tracks the number of pages, not the number of rows.
assert seq_idx.page_loads <= 2 * N / (PAGE_SIZE / 2)
assert rand_idx.page_loads > 5 * seq_idx.page_loads

💡 Sequential keys touch the same last page over and over (1 page load per page-full). Random keys bounce all over the tree — every insert is a cache miss.

Real DBs are more sophisticated (multiple pages in memory, write buffers, etc.), but the principle holds: **random primary keys on a big table really do hurt write throughput**.


## ✅ BEST — Time-ordered IDs (sneak peek)

What if we could have **both**?

- Coordination-free (generate on any machine) ✅
- Globally unique ✅
- **Sortable by creation time** → new IDs always land at the end of the B-tree ✅

This is the promise of **UUIDv7**, **ULID**, **KSUID**, and **Snowflake**. We'll meet them all in notebook 2 and 3.

A time-ordered ID roughly looks like this:

```
[ timestamp (high bits) ][ random or counter (low bits) ]
```

Because the timestamp is the **most significant** part, sorting IDs as numbers (or lexicographically as strings) sorts them by time. Inserts go to the end of the index. Problem solved.


In [ ]:
# A tiny preview: a hand-rolled time-ordered ID.
# 48 bits for milliseconds since epoch + 16 bits of randomness.
import os, time

def toy_time_ordered_id() -> int:
    ms = int(time.time() * 1000) & ((1 << 48) - 1)
    rand16 = int.from_bytes(os.urandom(2), 'big')
    return (ms << 16) | rand16

sample = []
for _ in range(5):
    sample.append(toy_time_ordered_id())
    time.sleep(0.002)  # 2 ms apart so timestamps differ
for s in sample:
    print(f"{s}  (hex: {s:016x})")

print()
assert sample == sorted(sample), "time-ordered IDs must sort by creation order"
print("Sorted == creation order?", sample == sorted(sample))

## 🧭 Summary

| Approach | Coordination | Unique | Sortable | DB-friendly |
|---|---|---|---|---|
| **BAD** — DB auto-increment | needs central DB | ✅ | ✅ | ✅ but bottlenecked |
| **BETTER** — UUIDv4 | none | ✅ | ❌ | ❌ index fragmentation |
| **BEST** — Time-ordered (UUIDv7/ULID/KSUID/Snowflake) | none | ✅ | ✅ | ✅ |

### What's next

- **Notebook 2**: implement UUIDv4, UUIDv7, ULID, and KSUID from scratch and compare them.
- **Notebook 3**: implement Twitter's Snowflake ID generator and explore its clock-skew pitfall.
